# 01 · TodoListMiddleware 基础

学习目标：创建带待办工具的 Agent，观察真实工具调用及 `todos` 更新。

资料是固定虚构数据；只有两个读取工具。模型编写实施方案，绝不执行未来开发步骤。

每个 Notebook 独立运行。内核工作目录应为本章 `notebooks/`，选择项目 `.venv` 内核。

In [ ]:
from planning_lab import (
    initialize, live_model, TOOLS, SYSTEM_PROMPT, REQUEST, FOLLOWUP,
    read_requirements, read_constraints, observe_update, final_answer,
)
initialize()
# 保持 False 可从头执行资料与 schema 检查，不调用模型。
# 确认密钥后手动改成 True，再从头运行。本章不会使用模拟模型。
RUN_LIVE = False
print(read_requirements.invoke({}))
print(read_constraints.invoke({}))
for tool in TOOLS:
    print(tool.name, tool.args_schema.model_json_schema())

## 查看中间件提供的工具

`write_todos` 由中间件注册，不需要手动加入 `TOOLS`。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware

# 这里只检查中间件与工具定义，不需要模型或密钥。
planning = TodoListMiddleware()
print(planning.tools[0].name)
print(planning.tools[0].args_schema.model_json_schema())

## 真实实验（主动开启）

将上方 `RUN_LIVE` 改为 `True` 后从头执行；需要有效 DeepSeek 密钥，会产生调用费用。

Agent 创建和循环如下。`updates` 返回节点增量，不是完整 State。`trace` 只是本地观察记录；本例不配置 Checkpointer，不调用 `get_state()`。

In [ ]:
if RUN_LIVE:
    model = live_model()
    agent = create_agent(
        model=model, tools=TOOLS, system_prompt=SYSTEM_PROMPT,
        middleware=[TodoListMiddleware()],
    )
    trace = {'messages': [], 'snapshots': [], 'todos': []}
    for update in agent.stream(
        {'messages': [{'role': 'user', 'content': REQUEST}]},
        {'recursion_limit': 60},
        stream_mode='updates',
    ):
        observe_update(update, trace)
    print('最终待办：', trace['todos'])
    print('最终方案：', final_answer(trace))
else:
    print('已跳过真实模型实验；请手动设置 RUN_LIVE=True。')

## 怎样读结果

1. 找到真实的 `write_todos`、`read_requirements`、`read_constraints` 调用及对应工具返回。
2. 比较第一次与后续待办：是否增加或细化了验证码约束核对工作？不要预设条数或逐字文案。
3. 核对最终回答：手机号验证码、密码暂缓、SQLite、限流、过期、失败处理和待确认阈值是否保留？
4. 待办里的“分析验证码规则”可以完成；方案里的“实现验证码接口”仍是未来工作。

可能因模型行为没有出现预期更新。请保留并检查真实轨迹，不要手动补写待办冒充模型行为。

`completed` 是模型写入的状态，不能证明成果质量；最终方案仍需人工复核。